In [1]:
!pip install transformers pandas torch scikit-learn
!pip install nltk spacy word2number
!python -m spacy download en_core_web_sm
!pip install -U transformers

  Preparing metadata (setup.py) ... done
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5568 sha256=d2ad7f155a5f54cc761dc33607034f2c65f316de89c001b3abce7f8e5e9516d2
  Stored in directory: /root/.cache/pip/wheels/5b/79/fb/d25928e599c7e11fe4e00d32048cd74933f34a74c633d2aea6
Successfully built word2number
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 54.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 114.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transfor

In [ ]:
# --- Imports ---
import os
import random
import re
from nltk.corpus import stopwords, wordnet
import nltk
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from transformers import (
    RobertaTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    RobertaModel,
    TrainerCallback
)
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# --- Setup ---
os.environ["WANDB_DISABLED"] = "true"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
KEEP_WORDS = {'not', 'no', 'shall', 'must', 'should', 'will', 'cannot', 'never', 'only'}
stop_words = set(stopwords.words('english')) - KEEP_WORDS

# ===============================
# ROBERTA WITH AUGMENTATION - RAW REQUIREMENT TEXT
# ===============================

def get_synonyms(word):
    clean_word = re.sub(r'[^a-zA-Z]', '', str(word).lower())
    if not clean_word:
        return []
    synonyms = set()
    for syn in wordnet.synsets(clean_word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != clean_word:
                synonyms.add(synonym)
    return list(synonyms)

def synonym_replacement(text, n=1):
    words = str(text).split()
    new_words = words.copy()
    random_word_list = list(set([
        word for word in words
        if re.sub(r'[^a-zA-Z]', '', word).lower() not in stop_words and len(word) > 3
    ]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)

def phrase_insertion(text, n=1):
    phrases = ["the system shall", "must be able to", "should ensure that"]
    words = str(text).split()
    if not words:
        return str(text)
    for _ in range(n):
        add_phrase = random.choice(phrases)
        random_idx = random.randint(0, len(words)-1)
        words.insert(random_idx, add_phrase)
    return ' '.join(words)

def text_simplification(text):
    text = re.sub(r'\bthe system shall\b', '', text)
    text = re.sub(r'\bmust be able to\b', '', text)
    text = re.sub(r'\bshould ensure that\b', '', text)
    return text.strip()

def random_deletion(text, p=0.1):
    words = str(text).split()
    if len(words) <= 1:
        return str(text)
    remaining = [word for word in words if random.random() > p]
    return ' '.join(remaining) if remaining else random.choice(words)

def random_swap(text, n=1):
    words = str(text).split()
    if len(words) < 2:
        return str(text)
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    return ' '.join(words)

# --- Data Loading ---
DATA_URL = 'https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv'
data_path = DATA_URL

df = pd.read_csv(data_path)
print(f"Loaded PROMISE expanded dataset from: {data_path}")
df['model_text'] = df['RequirementText'].astype(str)
df = df[df['class'].map(df['class'].value_counts()) > 1].reset_index(drop=True)

# --- Label Encoding & Split ---
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class'])

train_df, val_test_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=SEED
)

val_df, test_df = train_test_split(
    val_test_df, test_size=0.5,
    stratify=val_test_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# --- Augmentation (training split only) ---
augmentation_targets = {
    "FT": 200,
    "L": 200,
    "MN": 200,
    "A": 200,
    "LF": 200,
    "SC": 200,
    "PO": 200,
    "PE": 200,
    "O": 200,
    "US": 200,
    "SE": 100,
}
augmented = []

for cls, min_required in augmentation_targets.items():
    class_df = train_df[train_df['class'] == cls]
    needed = max(0, min_required - len(class_df))
    if len(class_df) == 0 or needed == 0:
        continue
    for _ in range(needed):
        row = class_df.sample(n=1, random_state=random.randint(0, 10**9)).iloc[0]
        text = row['model_text']
        aug_text = synonym_replacement(text, 1)
        aug_text = phrase_insertion(aug_text, 1)
        aug_text = text_simplification(aug_text)
        aug_text = random_swap(aug_text, 1)
        aug_text = random_deletion(aug_text, p=0.1)
        augmented.append({
            'ProjectID': row.get('ProjectID', None),
            'RequirementText': aug_text,
            'model_text': aug_text,
            'class': cls,
            'label': row['label'],
        })

df_aug = pd.DataFrame(augmented)
train_aug_df = pd.concat([train_df, df_aug], ignore_index=True) if len(df_aug) > 0 else train_df.copy()
train_aug_df = train_aug_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Original train distribution:", dict(train_df['class'].value_counts()))
print("Augmented train distribution:", dict(train_aug_df['class'].value_counts()))
print("Validation distribution:", dict(val_df['class'].value_counts()))
print("Test distribution:", dict(test_df['class'].value_counts()))

train_texts = train_aug_df['model_text'].astype(str).tolist()
train_labels = train_aug_df['label'].astype(int).tolist()
val_texts = val_df['model_text'].astype(str).tolist()
val_labels = val_df['label'].astype(int).tolist()
test_texts = test_df['model_text'].astype(str).tolist()
test_labels = test_df['label'].astype(int).tolist()

# --- Tokenization (RoBERTa ONLY) ---
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize(tokenizer, texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=128)

train_enc = tokenize(roberta_tokenizer, train_texts)
val_enc   = tokenize(roberta_tokenizer, val_texts)
test_enc  = tokenize(roberta_tokenizer, test_texts)

# --- Dataset ---
class RobertaOnlyDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.enc['input_ids'][idx]),
            'attention_mask': torch.tensor(self.enc['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

train_dataset = RobertaOnlyDataset(train_enc, train_labels)
val_dataset   = RobertaOnlyDataset(val_enc, val_labels)
test_dataset  = RobertaOnlyDataset(test_enc, test_labels)

# --- Model ---
class RobertaOnlyClassifier(torch.nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(768, num_labels)

        # Freeze lower transformer layers to reduce overfitting on short requirement texts.
        for name, param in self.roberta.named_parameters():
            if name.startswith("encoder.layer.") and int(name.split(".")[2]) < 8:
                param.requires_grad = False

        trainable_params = sum(param.numel() for param in self.parameters() if param.requires_grad)
        frozen_params = sum(param.numel() for param in self.parameters() if not param.requires_grad)
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Frozen parameters: {frozen_params:,}")

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        output = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(self.dropout(output))
        loss = torch.nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits}

# --- Metrics ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': p_w,
        'recall': r_w,
        'f1': f1_w,
        'precision_macro': p_m,
        'recall_macro': r_m,
        'macro_F1': f1_m,
    }


def compute_class_weights_from_labels(labels):
    label_array = np.array(labels, dtype=int)
    classes = np.arange(len(label_encoder.classes_))
    weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=label_array,
    )
    return torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        loss = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- Accuracy Callback ---
class AccuracyPlotCallback(TrainerCallback):
    def __init__(self):
        self.train_accuracies = []
        self.val_accuracies = []
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, args, state, control, **kwargs):
        train_metrics = trainer.evaluate(train_dataset)
        val_metrics = trainer.evaluate(val_dataset)
        self.train_accuracies.append(train_metrics["eval_accuracy"])
        self.val_accuracies.append(val_metrics["eval_accuracy"])
        self.train_losses.append(train_metrics["eval_loss"])
        self.val_losses.append(val_metrics["eval_loss"])

accuracy_callback = AccuracyPlotCallback()

# --- Training ---
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=30,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.02,
    learning_rate=1e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_F1",
    greater_is_better=True,
    save_total_limit=2,
    gradient_accumulation_steps=2,
    fp16=True,
    report_to="none",
)

model = RobertaOnlyClassifier(num_labels=len(label_encoder.classes_))

weight_tensor = compute_class_weights_from_labels(train_labels)
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    class_weights=weight_tensor,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[accuracy_callback]
)

trainer.train()

# --- Evaluation ---
test_results = trainer.predict(test_dataset)
test_preds = test_results.predictions.argmax(-1)

print(classification_report(
    test_labels,
    test_preds,
    target_names=label_encoder.classes_
))

# --- Confusion Matrix ---
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(10,7))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            cmap='Blues')
plt.show()


In [ ]:
# --- Imports ---
import os
import random
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from transformers import (
    RobertaTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    RobertaModel,
    TrainerCallback
)
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# --- Setup ---
os.environ["WANDB_DISABLED"] = "true"
SPLIT_SEED = 42
TRAIN_SEEDS = [42, 123, 2024]
SEED = SPLIT_SEED

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SPLIT_SEED)

# ===============================
# ROBERTA WITHOUT AUGMENTATION - RAW REQUIREMENT TEXT
# ===============================

# --- Data Loading ---
DATA_URL = 'https://raw.githubusercontent.com/egbertbritannia-cpu/rbl-requirements-classification/main/data/exp/promise_exp.csv'
data_path = DATA_URL

df = pd.read_csv(data_path)
print(f"Loaded PROMISE expanded dataset from: {data_path}")
df['model_text'] = df['RequirementText'].astype(str)
df = df[df['class'].map(df['class'].value_counts()) > 1].reset_index(drop=True)

# --- Label Encoding & Split ---
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class'])

train_df, val_test_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=SPLIT_SEED
)

val_df, test_df = train_test_split(
    val_test_df, test_size=0.5,
    stratify=val_test_df['label'], random_state=SPLIT_SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train distribution (no augmentation):", dict(train_df['class'].value_counts()))
print("Validation distribution:", dict(val_df['class'].value_counts()))
print("Test distribution:", dict(test_df['class'].value_counts()))

train_texts = train_df['model_text'].astype(str).tolist()
train_labels = train_df['label'].astype(int).tolist()
val_texts = val_df['model_text'].astype(str).tolist()
val_labels = val_df['label'].astype(int).tolist()
test_texts = test_df['model_text'].astype(str).tolist()
test_labels = test_df['label'].astype(int).tolist()

# --- Tokenization (RoBERTa ONLY) ---
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize(tokenizer, texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=128)

train_enc = tokenize(roberta_tokenizer, train_texts)
val_enc   = tokenize(roberta_tokenizer, val_texts)
test_enc  = tokenize(roberta_tokenizer, test_texts)

# --- Dataset ---
class RobertaOnlyDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.enc['input_ids'][idx]),
            'attention_mask': torch.tensor(self.enc['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

    def __len__(self):
        return len(self.labels)

train_dataset = RobertaOnlyDataset(train_enc, train_labels)
val_dataset   = RobertaOnlyDataset(val_enc, val_labels)
test_dataset  = RobertaOnlyDataset(test_enc, test_labels)

# --- Model ---
class RobertaOnlyClassifier(torch.nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(768, num_labels)

        # Freeze lower transformer layers to reduce overfitting on short requirement texts.
        for name, param in self.roberta.named_parameters():
            if name.startswith("encoder.layer.") and int(name.split(".")[2]) < 8:
                param.requires_grad = False

        trainable_params = sum(param.numel() for param in self.parameters() if param.requires_grad)
        frozen_params = sum(param.numel() for param in self.parameters() if not param.requires_grad)
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Frozen parameters: {frozen_params:,}")

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        output = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(self.dropout(output))
        loss = torch.nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits}

# --- Metrics ---
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': p_w,
        'recall': r_w,
        'f1': f1_w,
        'precision_macro': p_m,
        'recall_macro': r_m,
        'macro_F1': f1_m,
    }


def compute_class_weights_from_labels(labels):
    label_array = np.array(labels, dtype=int)
    classes = np.arange(len(label_encoder.classes_))
    weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=label_array,
    )
    return torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        loss = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )(logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- Accuracy Callback ---
class AccuracyPlotCallback(TrainerCallback):
    def __init__(self):
        self.train_accuracies = []
        self.val_accuracies = []
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, args, state, control, **kwargs):
        train_metrics = trainer.evaluate(train_dataset)
        val_metrics = trainer.evaluate(val_dataset)
        self.train_accuracies.append(train_metrics["eval_accuracy"])
        self.val_accuracies.append(val_metrics["eval_accuracy"])
        self.train_losses.append(train_metrics["eval_loss"])
        self.val_losses.append(val_metrics["eval_loss"])

# --- 3-seed runner: RoBERTa without augmentation ---

def summarize_seed_results(results_df, metric_columns):
    summary = results_df[metric_columns].agg(['mean', 'std']).T
    summary['mean ± std'] = summary.apply(
        lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}",
        axis=1,
    )
    return summary

def run_roberta_no_augmentation_seed(run_seed):
    global trainer
    set_all_seeds(run_seed)
    accuracy_callback = AccuracyPlotCallback()

    training_args = TrainingArguments(
        output_dir=f'./results/roberta_no_augmentation_seed_{run_seed}',
        num_train_epochs=30,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        warmup_ratio=0.1,
        weight_decay=0.02,
        learning_rate=1e-5,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_macro_F1",
        greater_is_better=True,
        save_total_limit=2,
        gradient_accumulation_steps=2,
        fp16=True,
        report_to="none",
        seed=run_seed,
        data_seed=run_seed,
    )

    start_time = time.perf_counter()
    model = RobertaOnlyClassifier(num_labels=len(label_encoder.classes_))
    weight_tensor = compute_class_weights_from_labels(train_labels)
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        class_weights=weight_tensor,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[accuracy_callback],
    )

    trainer.train()

    test_results = trainer.predict(test_dataset)
    runtime_seconds = time.perf_counter() - start_time
    test_preds = test_results.predictions.argmax(-1)
    metrics = compute_metrics(test_results)
    metrics["seed"] = run_seed
    metrics["runtime_seconds"] = runtime_seconds

    print(f"RoBERTa no augmentation | seed={run_seed} | runtime_seconds={runtime_seconds:.2f}")
    print(classification_report(
        test_labels,
        test_preds,
        target_names=label_encoder.classes_,
        zero_division=0,
    ))
    return metrics


roberta_no_aug_seed_results = [
    run_roberta_no_augmentation_seed(run_seed)
    for run_seed in TRAIN_SEEDS
]
roberta_no_aug_results_df = pd.DataFrame(roberta_no_aug_seed_results)
metric_columns = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "precision_macro",
    "recall_macro",
    "macro_F1",
    "runtime_seconds",
]
roberta_no_aug_summary = summarize_seed_results(roberta_no_aug_results_df, metric_columns)

print("RoBERTa no augmentation seed-level results:")
print(roberta_no_aug_results_df)
print("RoBERTa no augmentation mean ± std:")
print(roberta_no_aug_summary)
